In [ ]:
import sys
from contextlib import redirect_stdout 
import io

sys.path.append('../')
sys.path.append('../../src')

In [ ]:
import pandas as pd

df = pd.read_parquet('../ssi_all_features_30_days.parquet')
df.shape

In [ ]:
before_delivery_med_features = df.loc[df['description'] == 'before_delivery_medadmin_ingredient', 'feature_name'].unique().tolist()
after_delivery_med_features = df.loc[df['description'] == 'after_delivery_medadmin_ingredient', 'feature_name'].unique().tolist()
or_timings_features = df.loc[df['description'] == 'or_timings', 'feature_name'].unique().tolist()
len(before_delivery_med_features), len(after_delivery_med_features), len(or_timings_features)

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted

In [ ]:
from preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted)

In [ ]:
df_pivoted_preprocessed.isna().sum().sort_values(ascending=False)

In [ ]:
from feature_filter import feature_filter

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_filtered = feature_filter(df_pivoted_preprocessed, skip_medadmin_ingredient=True)

del df_pivoted_preprocessed

In [ ]:
df_pivoted_filtered.isna().sum().sort_values(ascending=False)

In [ ]:
before_delivery_med_features = [x for x in before_delivery_med_features if x in df_pivoted_filtered.columns]
after_delivery_med_features = [x for x in after_delivery_med_features if x in df_pivoted_filtered.columns]
or_timings_features = [x for x in df_pivoted_filtered.columns if x.startswith('or_timings')]
len(before_delivery_med_features), len(after_delivery_med_features), len(or_timings_features)

In [ ]:
med_features = [x for x in df_pivoted_filtered.columns if 'medadmin_ingredient' in x]
len(med_features)  # this has more since it has _isna features in it

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

clf = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42),
                   param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]},
                  #  param_grid= = {'min_samples_split': [500]} 
                   verbose=3,
                   cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   scoring=['f1', 'roc_auc'],
                   refit='roc_auc')

### 1 pcornet features only

In [ ]:
sys.path.insert(0, '../../')

from src.model_training import nested_cv_evaluation_feature_selection

In [ ]:
pcornet_features = [x for x in df_pivoted_filtered.columns \
                    if x not in or_timings_features and x != 'target_variable' and x != 'birthid' and x != 'mother_patid']
len(pcornet_features)

### 2. med features only using start time

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
        if x.endswith('earliest_minute'):
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

In [ ]:
X = df_pivoted_filtered[pcornet_features_subset]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_2 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

### 3. pcornet subset without start time and or timings

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)
print(len(pcornet_features_subset))

In [ ]:
X = df_pivoted_filtered[pcornet_features_subset + or_timings_features]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_3 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

### 4 only has med???

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

X = df_pivoted_filtered[pcornet_features_subset]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_4 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

### 5, earliest + or timings

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
        if x.endswith('earliest_minute'):
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

X = df_pivoted_filtered[pcornet_features_subset + or_timings_features]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_5 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

tmp1 = metrics_4
tmp1['test'] = 'Baseline EHR features'

tmp2 = metrics_3
tmp2['test'] = 'Baseline EHR + \nintraoperative timing features'

tmp3 = metrics_2
tmp3['test'] = 'Baseline EHR + \nmedication administration timing features'

tmp4 = metrics_5
tmp4['test'] = 'all features'

tmp = pd.concat([tmp1, tmp2, tmp3, tmp4], axis=0)
tmp.columns.name = ''

sns.barplot(tmp, x='roc_auc', y='test')
plt.xlim([0.60, 0.72])
plt.ylabel('')